## process_climate_normals
Extracts the NOAA Climate Normals tarball landed in `{catalog}.raw.climate_normals` and normalizes the 15,616 **ragged** per-station CSVs (508-2,140 cols each) into one tidy Parquet dataset under `RAW_CLIMATE_NORMALS_PROCESSED` (`.../climate_normals/_processed/normals_stations/`), keeping only the selected columns (all `StringType`, Bronze fidelity).

Runs after `download_weather_sources` in the `weather_download` job. The future climate-normals Bronze loader reads this `_processed` dataset.

Ragged-safe by design: each station file is parsed by HEADER NAME (`csv.DictReader`), so a station missing a measure null-fills instead of misaligning. Write strategy: **overwrite** (full snapshot, idempotent). Design: `weather_sources_download_design.md` Sec 4.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: RAW_CLIMATE_NORMALS, RAW_CLIMATE_NORMALS_PROCESSED,
# NORMALS_TARBALL, StepLog, AUDIT, PIPELINE_RUN_ID, Utils, spark, dbutils, StructType,
# StructField, StringType. The selected-column lists live in ONE place (CLAUDE.md Sec 6).
import os
import csv
import shutil
import tarfile
import tempfile

from climate_normals_columns import IDENTITY_COLS, MEASURE_COLS

STEP_SEQUENCE = 2

# Full selected column set: identity + measures + each measure's comp_flag_/years_ QC
# companions. Built here so the companion names have ONE definition (CLAUDE.md Sec 6).
COMPANION_PREFIXES = ("comp_flag", "years")
SELECTED_COLUMNS = list(IDENTITY_COLS) + list(MEASURE_COLS) + [
    f"{prefix}_{measure}"
    for measure in MEASURE_COLS
    for prefix in COMPANION_PREFIXES
]

In [ ]:
# Open the pipeline_step_log row (RUNNING). target_table=None: this step writes a
# _processed Parquet dataset to a Volume, not a managed table.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = None,
)
print(f"process_climate_normals: step_log_id={step.step_log_id}")

In [ ]:
# Extract + normalize. The Volume FUSE mount is unreliable for tarfile's random-access
# (seeking) reads, so copy the tarball to local scratch first, then untar from there. All
# processing is pure-Python on the driver (small data: ~15k stations); Spark is used only
# for the typed Parquet write at the end.
scratch_tarball = None
try:
    tarball_volume_path = f"{RAW_CLIMATE_NORMALS}{NORMALS_TARBALL}"
    scratch_tarball = os.path.join(tempfile.gettempdir(), NORMALS_TARBALL)
    shutil.copy(tarball_volume_path, scratch_tarball)   # Volume -> local (seekable)

    station_value_rows = []
    station_files_seen = 0
    with tarfile.open(scratch_tarball, "r:gz") as tarball:
        for member in tarball:
            if not member.name.endswith(".csv"):
                continue
            station_files_seen += 1
            # Each station file is a single-row CSV. Parse by HEADER NAME (DictReader) so
            # columns key by name, not position — what makes the ragged 508-vs-2,140-column
            # schema safe. Take that one row.
            station = next(csv.DictReader(tarball.extractfile(member).read().decode().splitlines()))
            if not station["STATION"].startswith("US"):   # US stations only (design Sec 5)
                continue
            # Keep only the wanted columns in schema order; a column absent from this
            # station (ragged) or empty null-fills.
            station_value_rows.append(
                tuple((station.get(column) or None) for column in SELECTED_COLUMNS)
            )

    if not station_value_rows:
        raise AssertionError(
            f"process_climate_normals: no US station rows extracted from {tarball_volume_path}"
        )

    # All-StringType schema, columns in SELECTED_COLUMNS order (Bronze does no casting).
    processed_schema = StructType(
        [StructField(column, StringType(), True) for column in SELECTED_COLUMNS]
    )
    processed_df = spark.createDataFrame(station_value_rows, schema=processed_schema)
    processed_df.write.mode("overwrite").parquet(RAW_CLIMATE_NORMALS_PROCESSED)

    step.rows_read    = station_files_seen
    step.rows_written = len(station_value_rows)
    step.succeed()
    print(f"process_climate_normals: {station_files_seen:,} station files -> "
          f"{len(station_value_rows):,} US rows x {len(SELECTED_COLUMNS)} cols "
          f"-> {RAW_CLIMATE_NORMALS_PROCESSED}")
except Exception as e:
    step.fail(e); raise
finally:
    if scratch_tarball and os.path.exists(scratch_tarball):
        os.remove(scratch_tarball)   # scratch cleanup